# ジツリス e5-base GPU 学習（Colab用）

このノートは、公開リポジトリの学習コードをColabのGPUランタイムで実行するための準備です。GoogleアカウントでColabにログインし、**ランタイム → ランタイムのタイプを変更 → T4 GPU（表示される場合）**を選んでください。無料枠のGPUは空き・利用制限により割り当てられないことがあります。

**注意:** Kaggle側の同じ学習が完了または明確に終了した後に使ってください。重複実行防止のため、学習許可フラグは既定で無効です。未見セットは読みません。評価はコード既定の固定5試験です。

In [ ]:
!git clone --depth 1 https://github.com/hehohahifu83-cmyk/jitsuris-train.git /content/jitsuris-train
%cd /content/jitsuris-train
!python -m pip install -q numpy scipy scikit-learn transformers sentencepiece

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPUが割り当てられていません。Colabのランタイム設定でGPUを選んでから、再接続してください。')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
RESULTS_ROOT = Path('/content/drive/MyDrive/jitsuris-e5base-results')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('保存先:', RESULTS_ROOT)

## 実行ガード

Kaggle側の状態を確認し、学習中でないことを確かめた場合だけ、下の2つの値を`True`にしてください。まず種1を実行し、完了後に`SEED = 2`へ変更して同じセルを実行します。

In [ ]:
KAGGLE_CLEARED = False  # Kaggleの学習が終わった/終了したと確認するまでFalse
RUN_TRAINING = False    # 実際に開始するときだけTrue
SAVE_MODEL = False      # Trueにすると約1.1GBのモデルもDriveへ保存
SEED = 1                # 1を完了させてから2へ

In [ ]:
import os, shutil, subprocess, sys
if not (KAGGLE_CLEARED and RUN_TRAINING):
    print('停止中です。Kaggleの状態確認後、必要な場合だけ両方のガードをTrueにしてください。')
elif SEED not in (1, 2):
    raise ValueError('SEEDは1か2を指定してください')
else:
    env = os.environ.copy()
    env.update({
        'FT_BS':'32', 'FT_ACC':'2', 'FT_CKPT':'0', 'FT_SORT':'1',
        'FT_PAD8':'1', 'FT_FUSED':'1', 'FT_CTX_UNIT':'1',
        'FT_FREEZE_EMB':'1', 'FT_SEED':str(SEED), 'TOKENIZERS_PARALLELISM':'false'
    })
    subprocess.run([sys.executable, 'finetune_exp.py', 'intfloat/multilingual-e5-base', '2'], env=env, check=True)
    dest = RESULTS_ROOT / f'seed-{SEED}'
    dest.mkdir(parents=True, exist_ok=True)
    for p in Path('.').glob('probs_*.json'):
        shutil.copy2(p, dest / p.name)
    for p in Path('.').glob('errors_*.json'):
        shutil.copy2(p, dest / p.name)
    if Path('FINETUNE_RESULTS.md').exists():
        shutil.copy2('FINETUNE_RESULTS.md', dest / 'FINETUNE_RESULTS.md')
    if SAVE_MODEL and Path('ft_out').exists():
        model_dest = dest / 'ft_out'
        if model_dest.exists(): shutil.rmtree(model_dest)
        shutil.copytree('ft_out', model_dest)
    print('保存完了:', dest)

## 条件と注意

- Kaggleの予定実験と同じe5-base・2周・UNIT文脈・埋め込み固定の条件です。Colab側では種を1本ずつ実行します。
- GPUが割り当てられない場合は学習セルを実行せず、同じ依頼を二重に送らないでください。
- 無料GPUの利用時間・種類・空きは保証されません。切断に備え、集計結果はGoogle Driveへ保存します。